<a id="capa"></a>
# Atividade 2.6 — Exemplos de Memória Cache

**Docente:** Claudomiro de Souza Sales Junior

**Discente:** Ana Beatriz Monteiro da Silva

**Curso:** Inteligência Artificial

---

Resolução dos exemplos de mapeamento de cache indicados nos slides **43**, **49** e **58**
da apresentação *Memória Cache*, usando como contexto os slides imediatamente anteriores.
Os três tipos de mapeamento são tratados: **direto**, **associativo** e **associativo por conjuntos**.


<a id="indice"></a>
## Índice

1. [Introdução: por que existe a memória cache](#introducao)
2. [Mapeamento direto](#sec1)
   - [Exemplo 5.2 — dimensionamento dos campos](#ex52)
   - [Exemplo 5.3 — separação de um endereço](#ex53)
   - [Exemplo 5.4 — leitura na cache (acerto ou falta)](#ex54)
3. [Mapeamento associativo](#sec2)
   - [Exemplo 5.5 (primeiro) — dimensionamento dos campos](#ex55a)
   - [Exemplo 5.5 (segundo) — comparação em paralelo](#ex55b)
   - [Exemplo 5.7 — separação de um endereço](#ex57)
4. [Mapeamento associativo por conjuntos](#sec3)
   - [Exemplo 5.8 — dimensionamento dos campos](#ex58)
   - [Exemplo 5.9 — separação de um endereço](#ex59)
   - [Exemplo 5.10 — leitura na cache (acerto ou falta)](#ex510)
5. [Comparação final entre os três mapeamentos](#a26-comparacao)
6. [Fontes, hipóteses, testes e limitações](#notas)


<a id="introducao"></a>
## 1. Introdução: por que existe a memória cache

O processador é muito mais rápido do que a memória principal (MP). Para não ficar parado
esperando os dados, coloca-se entre os dois uma memória pequena e veloz chamada **cache**.

A cache funciona por causa do **princípio da localidade**: quando um endereço é usado, é
muito provável que ele (e os vizinhos) sejam usados de novo logo em seguida. Por isso a MP
é dividida em **blocos** de células e, quando um bloco é necessário, ele inteiro é copiado
para uma **linha** da cache.

O problema central é: *em qual linha da cache cada bloco da MP pode ficar?* A regra que
responde isso é o **mapeamento**. Existem três técnicas, e cada uma divide o endereço de
forma diferente:

| Mapeamento | Onde o bloco pode ficar | Campos do endereço |
|---|---|---|
| Direto | em **uma** linha fixa | tag · linha · byte |
| Associativo | em **qualquer** linha | tag · byte |
| Por conjuntos | em qualquer linha de **um conjunto** | tag · conjunto · byte |

Em todos os exemplos a seguir a memória é a mesma (dados dos slides): MP com **2³² células**
(4 GB, célula de 1 byte), endereço de **32 bits**, **bloco de 64 células** (2⁶) e cache com
**1024 linhas** (2¹⁰).


<a id="sec1"></a>
## 2. Mapeamento direto

**Slide da atividade:** 43 — *"Exemplo 5.2, 5.3 e 5.4 · Fazer no Jupyter — passo a passo bem explicado"*.
**Contexto:** slides 35 a 42.

**Princípio de funcionamento.** No mapeamento direto cada bloco da MP só pode ocupar **uma**
linha da cache. A linha é escolhida por uma conta simples:

$$\text{linha} = (\text{número do bloco}) \bmod (\text{número de linhas})$$

O endereço é dividido em três campos — **tag · linha · byte**. O campo *byte* escolhe a célula
dentro do bloco, o campo *linha* aponta diretamente a linha da cache e a *tag* é guardada junto
com o dado para confirmar, na hora da leitura, se o bloco que está na linha é mesmo o procurado.
Como existe só uma linha possível, basta **um comparador** (barato), mas dois blocos que caem na
mesma linha não podem coexistir, o que aumenta as faltas (*miss*). Os slides 35 e 41 ilustram a
ideia com um exemplo pequeno de 5 bits (tag 2 · linha 2 · célula 1); abaixo usamos a configuração
completa de 32 bits.


<a id="ex52"></a>
### Exemplo 5.2 — dimensionamento dos campos

**Dados (slides 37, 38 e 39).** MP com 2³² células e endereço de 32 bits; bloco de 64 = 2⁶
células; cache com 1024 = 2¹⁰ linhas.

**Passo a passo.**
1. O bloco tem 2⁶ células → o campo **byte** precisa de **6 bits**.
2. A cache tem 2¹⁰ linhas → o campo **linha** precisa de **10 bits**.
3. A **tag** fica com o que sobra: 32 − 10 − 6 = **16 bits**.
4. Conferência: 16 + 10 + 6 = 32 bits (tamanho do endereço).


In [1]:
# Exemplo 5.2 - Mapeamento direto: tamanho dos campos do endereco
# Dados retirados dos slides 37, 38 e 39.

# Memoria principal
num_celulas = 2 ** 32          # 4 G celulas (4 GB); cada celula tem 1 byte
bits_endereco = 32             # o endereco da MP tem 32 bits

# Bloco
celulas_por_bloco = 2 ** 6     # cada bloco tem 64 celulas
bits_byte = 6                  # bits para escolher a celula dentro do bloco

# Quantos blocos existem na MP
num_blocos = num_celulas // celulas_por_bloco
print("Numero de blocos na MP:", num_blocos)
print("Conferindo a potencia: 2 ** 26 =", 2 ** 26)

# Cache (mapeamento direto)
num_linhas = 2 ** 10           # 1024 linhas
bits_linha = 10                # bits para escolher a linha

# A tag fica com o que sobra do endereco
bits_tag = bits_endereco - bits_linha - bits_byte

print("Bits de TAG  :", bits_tag)
print("Bits de LINHA:", bits_linha)
print("Bits de BYTE :", bits_byte)

# Conferencia 1: a soma dos campos tem que dar 32
soma = bits_tag + bits_linha + bits_byte
print("Soma dos campos:", soma, "bits")

# Conferencia 2: quantos blocos diferentes disputam a mesma linha
blocos_por_linha = num_blocos // num_linhas
print("Blocos que competem por cada linha:", blocos_por_linha)
print("Conferindo a potencia: 2 ** 16 =", 2 ** 16)


Numero de blocos na MP: 67108864
Conferindo a potencia: 2 ** 26 = 67108864
Bits de TAG  : 16
Bits de LINHA: 10
Bits de BYTE : 6
Soma dos campos: 32 bits
Blocos que competem por cada linha: 65536
Conferindo a potencia: 2 ** 16 = 65536


**Conclusão.** O endereço fica **tag(16) · linha(10) · byte(6)**, somando exatamente 32 bits.
Cada linha da cache é disputada por 2¹⁶ = 65 536 blocos diferentes da MP.

<a id="ex53"></a>
### Exemplo 5.3 — separação de um endereço

**Dados (slide 40).** Um endereço já dividido em campos: tag `0000000000000100`,
linha `0000110001`, byte `001000`.

**Passo a passo.**
1. Junto os três campos para formar o endereço de 32 bits.
2. Converto cada campo de binário para decimal.
3. Confiro por um segundo caminho, usando as fórmulas
   linha = bloco mod 1024 e tag = bloco ÷ 1024.


In [2]:
# Exemplo 5.3 - Mapeamento direto: separar um endereco nos campos
# Campos como aparecem no slide 40:
tag_bin   = "0000000000000100"   # 16 bits
linha_bin = "0000110001"         # 10 bits
byte_bin  = "001000"             # 6 bits

# 1) Junto os pedacos para formar o endereco completo
endereco_bin = tag_bin + linha_bin + byte_bin
print("Endereco completo (binario):", endereco_bin)
print("Quantidade de bits:", len(endereco_bin))

# 2) Converto cada campo de binario para decimal
tag = int(tag_bin, 2)
linha = int(linha_bin, 2)
byte = int(byte_bin, 2)
print("TAG   =", tag)
print("LINHA =", linha)
print("BYTE  =", byte)

# Valor decimal do endereco inteiro
endereco = int(endereco_bin, 2)
print("Endereco em decimal:", endereco)

# 3) Conferencia pela formula do mapeamento direto
celulas_por_bloco = 2 ** 6
num_linhas = 2 ** 10
numero_bloco = endereco // celulas_por_bloco
linha_formula = numero_bloco % num_linhas
tag_formula = numero_bloco // num_linhas
print("Numero do bloco:", numero_bloco)
print("Linha pela formula:", linha_formula)
print("Tag pela formula  :", tag_formula)

# Comparo os dois caminhos
if linha == linha_formula and tag == tag_formula:
    print("OK: a separacao por bits e a formula deram o mesmo resultado.")
else:
    print("ATENCAO: os resultados ficaram diferentes.")


Endereco completo (binario): 00000000000001000000110001001000
Quantidade de bits: 32
TAG   = 4
LINHA = 49
BYTE  = 8
Endereco em decimal: 265288
Numero do bloco: 4145
Linha pela formula: 49
Tag pela formula  : 4
OK: a separacao por bits e a formula deram o mesmo resultado.


**Conclusão.** O endereço vale 265 288 em decimal e corresponde ao **bloco 4145**, que no
mapeamento direto cai sempre na **linha 49** com **tag 4**; o byte pedido dentro do bloco é o 8.
Os dois métodos concordaram.

<a id="ex54"></a>
### Exemplo 5.4 — leitura na cache (acerto ou falta)

**Dados (slide 42).** A UCP envia um endereço de leitura: tag `0000000000000100`,
linha `0000011001`, byte `001000`. No slide, a linha 25 da cache guarda a tag 4, e a
comparação dá **4 = 4** (acerto).

**Passo a passo.**
1. Separo o endereço em tag (4), linha (25) e byte (8).
2. Vou direto à linha 25 (no direto só existe uma linha possível).
3. Comparo a tag pedida com a tag guardada nessa linha: se forem iguais é **acerto**.
4. Para reforçar, testo outro endereço que cai na mesma linha 25 mas com outra tag.


In [3]:
# Exemplo 5.4 - Mapeamento direto: leitura na cache (acerto ou falta)
# Endereco do slide 42:
tag_bin   = "0000000000000100"   # tag 4
linha_bin = "0000011001"         # linha 25
byte_bin  = "001000"             # byte 8

tag_pedido   = int(tag_bin, 2)
linha_pedida = int(linha_bin, 2)
byte_pedido  = int(byte_bin, 2)
print("Endereco pedido pela UCP -> tag:", tag_pedido, "| linha:", linha_pedida, "| byte:", byte_pedido)

# Estado ilustrativo da cache: para cada linha guardo a tag que esta armazenada.
# (Exemplo para demonstrar a comparacao; o slide 42 mostra a linha 25 guardando a tag 4.)
num_linhas = 1024
tags_guardadas = []
for i in range(num_linhas):
    tags_guardadas.append(-1)     # -1 = linha vazia

tags_guardadas[25] = 4            # como no slide

# No mapeamento direto so ha UMA linha possivel: a linha pedida.
tag_na_linha = tags_guardadas[linha_pedida]
print("Tag guardada na linha", linha_pedida, ":", tag_na_linha)

if tag_na_linha == tag_pedido:
    print("ACERTO (hit): a tag bate, o dado ja esta na cache.")
else:
    print("FALTA (miss): a tag nao bate, o bloco precisa ser trazido da MP.")

# Segundo teste: outro endereco que cai na MESMA linha, com outra tag -> forca uma falta
print()
tag_pedido2 = 7
print("Segundo teste -> mesma linha 25, mas tag pedida =", tag_pedido2)
if tags_guardadas[25] == tag_pedido2:
    print("ACERTO (hit)")
else:
    print("FALTA (miss): a linha 25 guarda a tag", tags_guardadas[25], "e nao a", tag_pedido2)


Endereco pedido pela UCP -> tag: 4 | linha: 25 | byte: 8
Tag guardada na linha 25 : 4
ACERTO (hit): a tag bate, o dado ja esta na cache.

Segundo teste -> mesma linha 25, mas tag pedida = 7
FALTA (miss): a linha 25 guarda a tag 4 e nao a 7


**Conclusão.** Quando a tag pedida é igual à tag guardada na linha indicada, há **acerto**
(caso do slide). O segundo teste mostra a fraqueza do mapeamento direto: dois blocos com a mesma
linha e tags diferentes não cabem juntos, gerando **falta** por conflito.

<a id="sec2"></a>
## 3. Mapeamento associativo

**Slide da atividade:** 49 — *"Exemplo 5.5, 5.5 e 5.7 · Fazer no Jupyter — passo a passo bem explicado"*.
**Contexto:** slides 44 a 48.

> **Observação sobre a numeração (provável erro de digitação).** O slide 49 traz, escrito
> literalmente, **"Exemplo 5.5, 5.5 e 5.7"**: o número **5.5 aparece repetido** e o **5.6 não é
> citado**. Como pede a atividade, a numeração foi **preservada exatamente como no slide**
> (5.5, 5.5 e 5.7). O segundo 5.5 **não** foi trocado por 5.6, porque o material não traz
> nenhuma evidência de qual seria o enunciado de um "Exemplo 5.6".

**Princípio de funcionamento.** Aqui o bloco pode ir para **qualquer** linha da cache (slide 46).
Isso dá mais flexibilidade e reduz as faltas, mas custa caro: não existe campo de *linha*, então
a **tag passa a ser o número do bloco inteiro** e, na leitura, é preciso comparar essa tag com as
tags de **todas** as linhas ao mesmo tempo (comparação em paralelo — slides 47 e 48). O endereço
fica só com dois campos: **tag · byte**.


<a id="ex55a"></a>
### Exemplo 5.5 (primeiro) — dimensionamento dos campos

**Dados.** Mesma memória dos exemplos anteriores (MP 2³² células, bloco 2⁶ células).
Princípio nos slides 45 e 46.

**Passo a passo.**
1. O bloco continua com 2⁶ células → campo **byte** = **6 bits**.
2. Não há campo de linha; a tag identifica o bloco inteiro. Como existem 2²⁶ blocos,
   a **tag** = **26 bits**.
3. Conferência: 26 + 6 = 32 bits.


In [4]:
# Exemplo 5.5 (primeiro) - Mapeamento associativo: tamanho dos campos
num_celulas = 2 ** 32
bits_endereco = 32
celulas_por_bloco = 2 ** 6
bits_byte = 6

num_blocos = num_celulas // celulas_por_bloco       # 2 ** 26
print("Numero de blocos na MP:", num_blocos, "(2 ** 26 =", 2 ** 26, ")")

# No associativo NAO existe campo de linha (o bloco vai para qualquer linha),
# por isso a tag precisa identificar o bloco inteiro.
bits_tag = bits_endereco - bits_byte
print("Bits de TAG :", bits_tag)
print("Bits de BYTE:", bits_byte)
print("Soma dos campos:", bits_tag + bits_byte, "bits")
print("A tag tem o mesmo tamanho do numero do bloco?", bits_tag == 26)


Numero de blocos na MP: 67108864 (2 ** 26 = 67108864 )
Bits de TAG : 26
Bits de BYTE: 6
Soma dos campos: 32 bits
A tag tem o mesmo tamanho do numero do bloco? True


**Conclusão.** Sem campo de linha, o endereço fica **tag(26) · byte(6)**. A tag é grande
porque sozinha precisa identificar qualquer um dos 2²⁶ blocos da MP.

<a id="ex55b"></a>
### Exemplo 5.5 (segundo) — comparação em paralelo

**Dados (slides 47 e 48).** Cache pequena de 4 linhas e bloco procurado `0101` (em decimal, 5),
exatamente como no diagrama do slide. Cada linha guarda a tag (número do bloco).

**Passo a passo.**
1. Converto o bloco procurado `0101` para decimal.
2. Comparo esse valor com a tag de **todas** as linhas (uso um laço para imitar o paralelo).
3. Se alguma linha tiver a mesma tag, é **acerto**; senão, **falta**.


In [5]:
# Exemplo 5.5 (segundo) - Mapeamento associativo: comparacao em paralelo
# Reproduz a ideia dos slides 47 e 48 com uma cache pequena de 4 linhas.
bloco_procurado = int("0101", 2)     # 0101 -> 5
print("Bloco procurado:", bloco_procurado)

# Estado ilustrativo: 4 linhas, cada uma guarda a tag (numero do bloco).
tags_guardadas = [2, 5, 9, 13]       # linha0=2, linha1=5, linha2=9, linha3=13

# No associativo a tag e comparada com TODAS as linhas ao mesmo tempo.
# O laco abaixo imita essa comparacao em paralelo.
achou = False
linha_acerto = -1
for i in range(len(tags_guardadas)):
    if tags_guardadas[i] == bloco_procurado:
        achou = True
        linha_acerto = i

if achou:
    print("ACERTO (hit): o bloco esta na linha", linha_acerto)
else:
    print("FALTA (miss): o bloco nao esta em nenhuma linha")

# Segundo teste: procurar um bloco que nao esta na cache
print()
bloco_procurado2 = 7
achou2 = False
for i in range(len(tags_guardadas)):
    if tags_guardadas[i] == bloco_procurado2:
        achou2 = True
print("Procurando o bloco", bloco_procurado2)
if achou2:
    print("ACERTO (hit)")
else:
    print("FALTA (miss): nenhuma linha guarda o bloco", bloco_procurado2)


Bloco procurado: 5
ACERTO (hit): o bloco esta na linha 1

Procurando o bloco 7
FALTA (miss): nenhuma linha guarda o bloco 7


**Conclusão.** O bloco 5 foi encontrado (linha 1) → **acerto**; o bloco 7 não está em
nenhuma linha → **falta**. Como o bloco pode estar em qualquer linha, todas precisam ser
comparadas ao mesmo tempo, o que exige muitos comparadores.

<a id="ex57"></a>
### Exemplo 5.7 — separação de um endereço

**Dados.** Para mostrar como a divisão muda, uso o **mesmo endereço do Exemplo 5.3**
(265 288, do slide 40), agora no esquema associativo.

**Passo a passo.**
1. O byte é a parte de dentro do bloco: byte = endereço mod 64.
2. A tag é o número do bloco inteiro: tag = endereço ÷ 64.
3. Escrevo os campos em binário (tag com 26 bits, byte com 6) e junto de novo para conferir.


In [6]:
# Exemplo 5.7 - Mapeamento associativo: separar um endereco real
# Mesmo endereco do Exemplo 5.3 (slide 40), agora no esquema associativo.
endereco = 265288
celulas_por_bloco = 2 ** 6

# byte = parte de dentro do bloco ; tag = numero do bloco inteiro
byte = endereco % celulas_por_bloco
tag = endereco // celulas_por_bloco
print("BYTE =", byte)
print("TAG (numero do bloco) =", tag)

# Campos em binario: tag com 26 bits, byte com 6 bits
tag_bin = bin(tag)[2:].zfill(26)
byte_bin = bin(byte)[2:].zfill(6)
print("TAG  em binario (26 bits):", tag_bin)
print("BYTE em binario (6 bits) :", byte_bin)

endereco_bin = tag_bin + byte_bin
print("Endereco montado de novo :", endereco_bin, "(", len(endereco_bin), "bits )")

# Conferencia: juntar de volta tem que dar o endereco original
endereco_conferido = int(endereco_bin, 2)
print("Endereco conferido:", endereco_conferido)
if endereco_conferido == endereco:
    print("OK: bateu com o endereco original.")
else:
    print("ATENCAO: nao bateu.")


BYTE = 8
TAG (numero do bloco) = 4145
TAG  em binario (26 bits): 00000000000001000000110001
BYTE em binario (6 bits) : 001000
Endereco montado de novo : 00000000000001000000110001001000 ( 32 bits )
Endereco conferido: 265288
OK: bateu com o endereco original.


**Conclusão.** O mesmo endereço do Exemplo 5.3 agora tem **tag 4145** (o número do bloco
inteiro) e **byte 8**, sem campo de linha. Repare que a tag associativa (4145) é justamente a
junção da tag (4) com a linha (49) do mapeamento direto: 4 × 1024 + 49 = 4145.

<a id="sec3"></a>
## 4. Mapeamento associativo por conjuntos

**Slide da atividade:** 58 — *"Exemplos 5.8, 5.9 e 5.10 · Fazer no Jupyter — passo a passo bem explicado"*.
**Contexto:** slides 50 a 57.

**Princípio de funcionamento.** É um **meio-termo** entre os dois anteriores (slide 50). A cache
é dividida em **conjuntos**; cada bloco está ligado a **um único conjunto**, mas dentro desse
conjunto ele pode ocupar **qualquer linha**. O conjunto é escolhido por:

$$\text{conjunto} = (\text{número do bloco}) \bmod (\text{número de conjuntos})$$

Assim a comparação de tags acontece só entre as linhas **daquele conjunto** (não a cache toda),
o que equilibra custo de hardware e taxa de faltas. O endereço fica com três campos:
**tag · conjunto · byte**. Nos slides 53 a 56 a cache de 1024 linhas é organizada em **32 conjuntos**
de **32 linhas** cada.


<a id="ex58"></a>
### Exemplo 5.8 — dimensionamento dos campos

**Dados (slides 53, 54, 55 e 56).** Mesma memória; cache com 1024 = 2¹⁰ linhas organizada em
32 = 2⁵ conjuntos.

**Passo a passo.**
1. Byte dentro do bloco → **6 bits**.
2. 32 conjuntos = 2⁵ → campo **conjunto** = **5 bits**.
3. Linhas por conjunto = 1024 ÷ 32 = 32.
4. A **tag** fica com o resto: 32 − 5 − 6 = **21 bits**.


In [7]:
# Exemplo 5.8 - Mapeamento por conjuntos: tamanho dos campos
# Dados dos slides 53, 54, 55 e 56.
num_celulas = 2 ** 32
bits_endereco = 32
celulas_por_bloco = 2 ** 6
bits_byte = 6
num_blocos = num_celulas // celulas_por_bloco       # 2 ** 26

num_linhas = 2 ** 10           # 1024 linhas no total
num_conjuntos = 2 ** 5         # 32 conjuntos
bits_conjunto = 5              # bits para escolher o conjunto

linhas_por_conjunto = num_linhas // num_conjuntos
print("Linhas por conjunto:", linhas_por_conjunto)

bits_tag = bits_endereco - bits_conjunto - bits_byte
print("Bits de TAG     :", bits_tag)
print("Bits de CONJUNTO:", bits_conjunto)
print("Bits de BYTE    :", bits_byte)
print("Soma dos campos :", bits_tag + bits_conjunto + bits_byte, "bits")

blocos_por_conjunto = num_blocos // num_conjuntos
print("Blocos que competem por cada conjunto:", blocos_por_conjunto)
print("Conferindo a potencia: 2 ** 21 =", 2 ** 21)


Linhas por conjunto:

 32
Bits de TAG     : 21
Bits de CONJUNTO: 5
Bits de BYTE    : 6
Soma dos campos : 32 bits
Blocos que competem por cada conjunto: 2097152
Conferindo a potencia: 2 ** 21 = 2097152


**Conclusão.** O endereço fica **tag(21) · conjunto(5) · byte(6)** = 32 bits. Cada conjunto
tem 32 linhas e é disputado por 2²¹ blocos da MP.

<a id="ex59"></a>
### Exemplo 5.9 — separação de um endereço

**Dados (slide 57).** Endereço dividido em campos: tag `000000000000000000100` (21 bits),
conjunto `10001` (5 bits) e byte `001000` (6 bits).

**Passo a passo.**
1. Junto os três campos para formar o endereço de 32 bits e converto cada um para decimal.
2. Confiro por outro caminho: conjunto = bloco mod 32 e tag = bloco ÷ 32.


In [8]:
# Exemplo 5.9 - Mapeamento por conjuntos: separar um endereco
# Campos como aparecem no slide 57:
tag_bin      = "0" * 18 + "100"   # 21 bits  (dezoito zeros + 100)
conjunto_bin = "10001"            # 5 bits
byte_bin     = "001000"           # 6 bits

endereco_bin = tag_bin + conjunto_bin + byte_bin
print("Endereco completo:", endereco_bin, "(", len(endereco_bin), "bits )")

# Conferindo os tamanhos de cada campo
print("Tamanho da tag:", len(tag_bin), "| conjunto:", len(conjunto_bin), "| byte:", len(byte_bin))

tag = int(tag_bin, 2)
conjunto = int(conjunto_bin, 2)
byte = int(byte_bin, 2)
print("TAG      =", tag)
print("CONJUNTO =", conjunto)
print("BYTE     =", byte)

endereco = int(endereco_bin, 2)
print("Endereco em decimal:", endereco)

# Conferencia pela formula
celulas_por_bloco = 2 ** 6
num_conjuntos = 2 ** 5
numero_bloco = endereco // celulas_por_bloco
conjunto_formula = numero_bloco % num_conjuntos
tag_formula = numero_bloco // num_conjuntos
print("Numero do bloco:", numero_bloco)
print("Conjunto pela formula:", conjunto_formula)
print("Tag pela formula     :", tag_formula)

if conjunto == conjunto_formula and tag == tag_formula:
    print("OK: separacao por bits e formula concordam.")
else:
    print("ATENCAO: resultados diferentes.")


Endereco completo: 00000000000000000010010001001000 ( 32 bits )
Tamanho da tag: 21 | conjunto: 5 | byte: 6
TAG      = 4
CONJUNTO = 17
BYTE     = 8
Endereco em decimal: 9288
Numero do bloco: 145
Conjunto pela formula: 17
Tag pela formula     : 4
OK: separacao por bits e formula concordam.


**Conclusão.** O endereço vale 9288 e corresponde ao **bloco 145**, que cai no
**conjunto 17** com **tag 4** e byte 8. Os dois métodos concordaram.

<a id="ex510"></a>
### Exemplo 5.10 — leitura na cache (acerto ou falta)

**Dados.** Uso o endereço do Exemplo 5.9: tag 4, conjunto 17, byte 8.

**Passo a passo.**
1. Vou ao conjunto 17 (o bloco está ligado a um único conjunto).
2. No hardware, a tag pedida é comparada em paralelo com as tags das 32 linhas; no código, um laço simula essa busca.
3. Se alguma linha tiver a tag, é **acerto**; senão, **falta**.
4. Faço também um teste com uma tag que não está no conjunto.


In [9]:
# Exemplo 5.10 - Mapeamento por conjuntos: leitura (acerto ou falta)
# Endereco do Exemplo 5.9: tag 4, conjunto 17, byte 8.
tag_pedido = 4
conjunto_pedido = 17
print("Procurando a tag", tag_pedido, "no conjunto", conjunto_pedido)

# Cada conjunto tem 32 linhas. Represento SO o conjunto 17 com uma lista de tags
# guardadas (exemplo ilustrativo). -1 significa linha vazia.
linhas_por_conjunto = 32
tags_do_conjunto = []
for i in range(linhas_por_conjunto):
    tags_do_conjunto.append(-1)

# Coloco algumas tags de exemplo dentro do conjunto 17
tags_do_conjunto[0] = 30
tags_do_conjunto[1] = 4      # esta linha guarda a tag procurada
tags_do_conjunto[2] = 11

# O laco abaixo simula a comparacao paralela feita pelo hardware.
achou = False
linha_acerto = -1
for i in range(linhas_por_conjunto):
    if tags_do_conjunto[i] == tag_pedido:
        achou = True
        linha_acerto = i

if achou:
    print("ACERTO (hit): tag encontrada na linha", linha_acerto, "do conjunto", conjunto_pedido)
else:
    print("FALTA (miss): a tag nao esta no conjunto", conjunto_pedido)

# Segundo teste: tag que nao esta no conjunto
print()
tag_pedido2 = 99
achou2 = False
for i in range(linhas_por_conjunto):
    if tags_do_conjunto[i] == tag_pedido2:
        achou2 = True
print("Procurando agora a tag", tag_pedido2, "no conjunto 17")
if achou2:
    print("ACERTO (hit)")
else:
    print("FALTA (miss): a tag", tag_pedido2, "nao esta no conjunto 17")


Procurando a tag 4 no conjunto 17
ACERTO (hit): tag encontrada na linha 1 do conjunto 17

Procurando agora a tag 99 no conjunto 17
FALTA (miss): a tag 99 nao esta no conjunto 17


**Conclusão.** A tag 4 foi achada na linha 1 do conjunto 17 → **acerto**; a tag 99 não
está no conjunto → **falta**. A busca olha só as 32 linhas do conjunto, e não a cache inteira.

<a id="a26-comparacao"></a>
## 5. Comparação final entre os três mapeamentos

Para enxergar a diferença, separamos **o mesmo endereço** (265 288, do slide 40, que é o
**bloco 4145**) nos três esquemas. A cache tem sempre o mesmo tamanho — 1024 linhas de 64 células
(64 KB); o que muda é a **liberdade de onde o bloco pode ficar** e o **tamanho dos campos**.


In [10]:
# Comparacao final: o MESMO endereco nos tres tipos de mapeamento
endereco = 265288
celulas_por_bloco = 2 ** 6
numero_bloco = endereco // celulas_por_bloco
byte = endereco % celulas_por_bloco
print("Endereco:", endereco, "| numero do bloco:", numero_bloco, "| byte:", byte)
print()

# 1) Direto: tag(16) | linha(10) | byte(6)
num_linhas = 2 ** 10
linha_d = numero_bloco % num_linhas
tag_d = numero_bloco // num_linhas
print("DIRETO        -> tag:", tag_d, "| linha fixa:", linha_d, "| byte:", byte)

# 2) Associativo: tag(26) | byte(6)  -> sem linha fixa
tag_a = numero_bloco
print("ASSOCIATIVO   -> tag:", tag_a, "| linha: qualquer uma | byte:", byte)

# 3) Por conjuntos: tag(21) | conjunto(5) | byte(6)
num_conjuntos = 2 ** 5
conjunto_c = numero_bloco % num_conjuntos
tag_c = numero_bloco // num_conjuntos
print("POR CONJUNTOS -> tag:", tag_c, "| conjunto:", conjunto_c, "| byte:", byte)

print()
print("Tamanho da cache nos tres casos: 1024 linhas x 64 celulas =", 1024 * 64, "celulas (64 KB)")


Endereco: 265288 | numero do bloco: 4145 | byte: 8

DIRETO        -> tag: 4 | linha fixa: 49 | byte: 8
ASSOCIATIVO   -> tag: 4145 | linha: qualquer uma | byte: 8
POR CONJUNTOS -> tag: 129 | conjunto: 17 | byte: 8

Tamanho da cache nos tres casos: 1024 linhas x 64 celulas = 65536 celulas (64 KB)


**Resumo comparativo.**

| Característica | Direto | Associativo | Por conjuntos |
|---|---|---|---|
| Onde o bloco pode ficar | 1 linha fixa | qualquer linha | qualquer linha de 1 conjunto |
| Campos do endereço (bits) | 16 · 10 · 6 | 26 · 6 | 21 · 5 · 6 |
| Comparadores na leitura | 1 | todas as linhas | linhas do conjunto |
| Custo / complexidade | menor | maior | intermediário |
| Faltas por conflito | mais | menos | intermediário |

Para o bloco 4145: no **direto** ele é obrigado a ficar na linha 49; no **associativo** pode ficar
em qualquer das 1024 linhas; no **por conjuntos** fica preso ao conjunto 17, mas livre entre as 32
linhas desse conjunto. O direto é o mais simples e barato, o associativo o mais flexível e caro, e
o por conjuntos é o equilíbrio entre os dois — por isso é o mais usado na prática.


<a id="notas"></a>
## 6. Fontes, hipóteses, testes e limitações

**Fontes (apresentação *Memória Cache*).**
- Mapeamento **direto** — slide **43** (enunciado da atividade); contexto nos slides **35–42**
  (formato do endereço no slide 39; endereços de exemplo nos slides 40 e 42).
- Mapeamento **associativo** — slide **49** (enunciado); contexto nos slides **44–48**
  (princípio nos slides 45–46; comparação em paralelo nos slides 47–48).
- Mapeamento **por conjuntos** — slide **58** (enunciado); contexto nos slides **50–57**
  (organização e formato nos slides 53–56; endereço de exemplo no slide 57).

**Hipóteses assumidas.**
- Cada célula da MP corresponde a 1 byte (os slides usam "célula" e "byte" para o mesmo campo de 6 bits).
- Os números de exemplo (5.2, 5.3, 5.4, 5.5, 5.7, 5.8, 5.9, 5.10) aparecem **apenas** nos slides de
  título 43, 49 e 58, referindo-se à numeração do material-base; os slides de figura não rotulam
  individualmente cada exemplo. Por isso, cada exemplo foi organizado seguindo a progressão natural
  das figuras (dimensionar campos → separar um endereço → leitura com acerto/falta), usando
  **exclusivamente os dados presentes nos slides**.
- Os estados de cache usados nas leituras (Exemplos 5.4, 5.5-segundo e 5.10) são **ilustrativos**,
  servindo só para demonstrar a comparação de tags; estão sinalizados como exemplo no código, no
  mesmo espírito do slide 42 (que mostra a linha 25 guardando a tag 4).

**Provável erro de digitação registrado.** O slide 49 escreve "Exemplo 5.5, 5.5 e 5.7": o número
**5.5 está repetido** e o **5.6 não aparece**. A numeração foi mantida como no slide; o segundo
5.5 **não** foi alterado para 5.6 por falta de evidência no material.

**Testes realizados (todos com construções básicas: variáveis, laços e `if/else`).**
- Soma dos campos = 32 bits nos três mapeamentos (Exemplos 5.2, 5.5-primeiro e 5.8). ✔
- Separação de endereço conferida por **dois métodos** (recorte de bits × fórmula bloco/linha/conjunto)
  nos Exemplos 5.3, 5.7 e 5.9 — todos com resultados iguais. ✔
- Leitura com **acerto** reproduzindo o slide e **falta** em teste de contraste (Exemplos 5.4 e 5.10);
  busca em paralelo no associativo (Exemplo 5.5-segundo). ✔
- Verificação das potências de 2 (2²⁶, 2¹⁶, 2²¹) impressa lado a lado com os valores calculados. ✔

**Limitações.**
- Não há, no material, um endereço de exemplo de 32 bits específico para o **associativo**; por isso
  o Exemplo 5.7 reaproveita, de forma transparente, o mesmo endereço do Exemplo 5.3, apenas mudando
  o esquema de divisão.
- A atividade é de cálculo e simulação em Python; não envolve circuito no Logisim.
